In [1]:
# ============================================
# 3일차 핵심 실습
# CSV 읽기 -> SQLite 적재 -> SQL 조회 -> 품질 점검 -> 로그 저장
# ============================================

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import pandas as pd
import sqlite3
from pathlib import Path
from datetime import datetime

In [4]:
# -------------------------------------------------
# 1. 폴더 설정
# -------------------------------------------------
base_dir = Path(".")
interim_dir = base_dir / "data" / "interim"
output_dir = base_dir / "data" / "output"

interim_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

In [5]:
# -------------------------------------------------
# 2. 로그 준비
# -------------------------------------------------
log_messages = []
start_time = datetime.now()

log_messages.append(f"[START] {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

In [6]:
# -------------------------------------------------
# 3. 2일차 결과물 불러오기
# -------------------------------------------------
input_file = interim_dir / "ai4i_enriched.csv"
df = pd.read_csv(input_file)

print("=" * 60)
print("2일차 결과물 불러오기")
print("=" * 60)
print("데이터 크기:", df.shape)
display(df.head())

log_messages.append(f"[READ] input_file={input_file}")
log_messages.append(f"[READ] rows={df.shape[0]}, cols={df.shape[1]}")

2일차 결과물 불러오기
데이터 크기: (10000, 26)


,udi,product_id,type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level,avg_daily_target,recommended_temp_range,maintenance_priority,high_wear_flag,failure_risk_note
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,low,0,medium grade,20,standard,145.0,300~310K,medium,0,normal
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal


In [7]:
# -------------------------------------------------
# 4. SQLite 연결
# -------------------------------------------------
db_file = output_dir / "manufacturing_pipeline.db"
conn = sqlite3.connect(db_file)

print("\nSQLite 연결 완료")
print("DB 파일:", db_file)

log_messages.append(f"[DB] connected={db_file}")


SQLite 연결 완료
DB 파일: data/output/manufacturing_pipeline.db


In [8]:
# -------------------------------------------------
# 5. 기존 테이블 삭제 후 적재
# -------------------------------------------------
conn.execute("DROP TABLE IF EXISTS manufacturing_data")
conn.commit()

df.to_sql("manufacturing_data", conn, if_exists="replace", index=False)

print("\nSQLite 적재 완료")
log_messages.append("[LOAD] table=manufacturing_data loaded")


SQLite 적재 완료


In [9]:
# -------------------------------------------------
# 6. 적재 후 row 수 확인
# -------------------------------------------------
row_count_df = pd.read_sql("SELECT COUNT(*) AS cnt FROM manufacturing_data", conn)
db_row_count = row_count_df.loc[0, "cnt"]

print("\n원본 행 수:", df.shape[0])
print("DB 행 수  :", db_row_count)

if df.shape[0] == db_row_count:
    print("행 수 일치")
    log_messages.append("[CHECK] row_count_match=True")
else:
    print("행 수 불일치")
    log_messages.append("[CHECK] row_count_match=False")



원본 행 수: 10000
DB 행 수  : 10000
행 수 일치


In [10]:
# -------------------------------------------------
# 7. 샘플 조회
# -------------------------------------------------
sample_df = pd.read_sql("SELECT * FROM manufacturing_data LIMIT 5", conn)

print("\nDB 샘플 조회")
display(sample_df)


DB 샘플 조회


,udi,product_id,type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level,avg_daily_target,recommended_temp_range,maintenance_priority,high_wear_flag,failure_risk_note
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,low,0,medium grade,20,standard,145.0,300~310K,medium,0,normal
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal


In [11]:
# -------------------------------------------------
# 8. SQL 조회 예시 1
# -------------------------------------------------
sql_1 = """
SELECT
    type,
    ROUND(AVG(tool_wear_min), 2) AS avg_tool_wear
FROM manufacturing_data
GROUP BY type
ORDER BY type
"""

result_1 = pd.read_sql(sql_1, conn)

print("\ntype별 평균 공구 마모")
display(result_1)


type별 평균 공구 마모


,type,avg_tool_wear
0,H,107.42
1,L,108.38
2,M,107.27


In [12]:
# -------------------------------------------------
# 9. SQL 조회 예시 2
# -------------------------------------------------
sql_2 = """
SELECT
    machine_failure,
    COUNT(*) AS cnt
FROM manufacturing_data
GROUP BY machine_failure
ORDER BY machine_failure
"""

result_2 = pd.read_sql(sql_2, conn)

print("\nmachine_failure별 건수")
display(result_2)


machine_failure별 건수


,machine_failure,cnt
0,0,9661
1,1,339


In [13]:
# -------------------------------------------------
# 10. SQL 조회 예시 3
# -------------------------------------------------
sql_3 = """
SELECT
    failure_risk_note,
    COUNT(*) AS cnt
FROM manufacturing_data
GROUP BY failure_risk_note
ORDER BY cnt DESC
"""

result_3 = pd.read_sql(sql_3, conn)

print("\nfailure_risk_note별 건수")
display(result_3)


failure_risk_note별 건수


,failure_risk_note,cnt
0,normal,9816
1,check_now,184


In [14]:
# -------------------------------------------------
# 11. 품질 점검 - 결측치
# -------------------------------------------------
null_counts = df.isnull().sum().reset_index()
null_counts.columns = ["column_name", "null_count"]

print("\n컬럼별 결측치 개수")
display(null_counts)


컬럼별 결측치 개수


,column_name,null_count
0,udi,0
1,product_id,0
2,type,0
3,air_temp_k,0
4,process_temp_k,0
5,rot_speed_rpm,0
6,torque_nm,0
7,tool_wear_min,0
8,machine_failure,0
9,twf,0


In [15]:
# -------------------------------------------------
# 12. 품질 점검 - 중복
# -------------------------------------------------
duplicate_count = df.duplicated().sum()

print("\n중복 행 개수:", duplicate_count)


중복 행 개수: 0


In [16]:
# -------------------------------------------------
# 13. 품질 점검 - 주요 수치형 범위
# -------------------------------------------------
range_check = pd.DataFrame({
    "column_name": ["air_temp_k", "process_temp_k", "rot_speed_rpm", "torque_nm", "tool_wear_min"],
    "min_value": [
        df["air_temp_k"].min(),
        df["process_temp_k"].min(),
        df["rot_speed_rpm"].min(),
        df["torque_nm"].min(),
        df["tool_wear_min"].min()
    ],
    "max_value": [
        df["air_temp_k"].max(),
        df["process_temp_k"].max(),
        df["rot_speed_rpm"].max(),
        df["torque_nm"].max(),
        df["tool_wear_min"].max()
    ]
})

print("\n주요 수치형 컬럼 최소/최대값")
display(range_check)


주요 수치형 컬럼 최소/최대값


,column_name,min_value,max_value
0,air_temp_k,295.3,304.5
1,process_temp_k,305.7,313.8
2,rot_speed_rpm,1168.0,2886.0
3,torque_nm,3.8,76.6
4,tool_wear_min,0.0,253.0


In [17]:
# -------------------------------------------------
# 14. 품질 점검 요약표 만들기
# -------------------------------------------------
quality_summary = pd.DataFrame({
    "check_item": [
        "source_row_count",
        "db_row_count",
        "row_count_match",
        "duplicate_count",
        "total_null_count"
    ],
    "check_result": [
        df.shape[0],
        db_row_count,
        df.shape[0] == db_row_count,
        duplicate_count,
        int(df.isnull().sum().sum())
    ]
})

print("\n품질 점검 요약")
display(quality_summary)


품질 점검 요약


,check_item,check_result
0,source_row_count,10000
1,db_row_count,10000
2,row_count_match,True
3,duplicate_count,0
4,total_null_count,2


In [18]:
# -------------------------------------------------
# 15. 품질 점검 결과 저장
# -------------------------------------------------
quality_file = output_dir / "quality_check_report.csv"
quality_summary.to_csv(quality_file, index=False, encoding="utf-8-sig")

print("\n품질 점검 결과 저장 완료:", quality_file)

log_messages.append(f"[QUALITY] duplicate_count={duplicate_count}")
log_messages.append(f"[QUALITY] total_null_count={int(df.isnull().sum().sum())}")
log_messages.append(f"[QUALITY] report_file={quality_file}")


품질 점검 결과 저장 완료: data/output/quality_check_report.csv


In [19]:
# -------------------------------------------------
# 16. 로그 종료 정보 추가
# -------------------------------------------------
end_time = datetime.now()
elapsed_seconds = (end_time - start_time).total_seconds()

log_messages.append(f"[END] {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
log_messages.append(f"[END] elapsed_seconds={elapsed_seconds:.2f}")

In [20]:
# -------------------------------------------------
# 17. 로그 파일 저장
# -------------------------------------------------
log_file = output_dir / "etl_run_log.txt"

with open(log_file, "w", encoding="utf-8") as f:
    for msg in log_messages:
        f.write(msg + "\n")

print("로그 파일 저장 완료:", log_file)

로그 파일 저장 완료: data/output/etl_run_log.txt


In [21]:
# -------------------------------------------------
# 18. DB 연결 종료
# -------------------------------------------------
conn.close()

print("\nSQLite 연결 종료")


SQLite 연결 종료


In [22]:
# -------------------------------------------------
# 19. 마무리 요약
# -------------------------------------------------
print("\n" + "=" * 60)
print("3일차 실습 요약")
print("=" * 60)
print("1) 2일차 통합 데이터를 다시 불러왔다.")
print("2) SQLite DB 파일을 만들고 연결했다.")
print("3) manufacturing_data 테이블에 데이터를 적재했다.")
print("4) SQL로 조회하여 적재 결과를 확인했다.")
print("5) 품질 점검 결과와 실행 로그를 파일로 저장했다.")


3일차 실습 요약
1) 2일차 통합 데이터를 다시 불러왔다.
2) SQLite DB 파일을 만들고 연결했다.
3) manufacturing_data 테이블에 데이터를 적재했다.
4) SQL로 조회하여 적재 결과를 확인했다.
5) 품질 점검 결과와 실행 로그를 파일로 저장했다.


In [23]:
# end